# carGO PH — Blockchain Data Retrieval & Cleaning
**Course:** MO-IT148 — Application Development and Emerging Technologies  
**Group:** NodeBlk

**Week:** 6 — Data Retrieval + Cleaning + Stats  
**Description:** Retrieves IoT sensor records from the IoTDataStorage smart contract on Ganache,  
cleans and enriches the data, and exports `5-iot_cleaned_data_v2.csv` for Tableau.

> **v2 additions:** `breach_delta_c`, `scan_hour`, `scan_day` computed columns;  
> `package_count`, `vehicle_id`, `driver_id`, `shipment_status`, `scheduled_delivery_dt`,  
> `actual_delivery_dt`, `delay_reason` joined from `1-shipment_registry_v2.csv`.  
> Final output: **25 columns** vs 14 in v1.

> ⚠️ Prerequisite: Ganache must be running with the contract deployed before executing Setup.

## 1. Setup

In [20]:
from web3 import Web3
import pandas as pd
import numpy as np
import json

with open('../contracts/IoTDataStorage_compData.json') as f:
    comp_data = json.load(f)

abi = json.loads(comp_data['metadata'])['output']['abi']

CONTRACT_ADDRESS  = Web3.to_checksum_address("0x8bA344300911947b128474f05Fb1d95c3120d152")  # ← update after redeploy

web3              = Web3(Web3.HTTPProvider("http://127.0.0.1:7545"))
logistics_contract = web3.eth.contract(address=CONTRACT_ADDRESS, abi=abi)
web3.eth.default_account = web3.eth.accounts[0]

print("Connected:",          web3.is_connected())
print("GPS records:",        logistics_contract.functions.gpsRecordCount().call())
print("Temp records:",       logistics_contract.functions.tempRecordCount().call())
print("RFID records:",       logistics_contract.functions.rfidRecordCount().call())


Connected: True
GPS records: 400
Temp records: 240
RFID records: 226


## 2. Retrieval — pull all records from chain

In [21]:
# GPS
gps_count = logistics_contract.functions.gpsRecordCount().call()
gps_rows  = []
for i in range(gps_count):
    r = logistics_contract.functions.gpsRecords(i).call()
    gps_rows.append({"timestamp": r[0], "rfid_tag": r[1], "device_id": r[2],
                     "latitude": r[3], "longitude": r[4], "sensor_type": "GPS"})
gps_records_df = pd.DataFrame(gps_rows)

# Temperature
temp_count = logistics_contract.functions.tempRecordCount().call()
temp_rows  = []
for i in range(temp_count):
    r = logistics_contract.functions.tempRecords(i).call()
    temp_rows.append({"timestamp": r[0], "rfid_tag": r[1], "device_id": r[2],
                      "temperature_raw": r[3], "sensor_type": "Temperature"})
temp_records_df = pd.DataFrame(temp_rows)

# RFID
rfid_count = logistics_contract.functions.rfidRecordCount().call()
rfid_rows  = []
for i in range(rfid_count):
    r = logistics_contract.functions.rfidRecords(i).call()
    rfid_rows.append({"timestamp": r[0], "rfid_tag": r[1], "device_id": r[2],
                      "scan_status": r[3], "sensor_type": "RFID"})
rfid_records_df = pd.DataFrame(rfid_rows)

print(f"GPS: {len(gps_records_df)} | Temp: {len(temp_records_df)} | RFID: {len(rfid_records_df)}")


GPS: 400 | Temp: 240 | RFID: 226


## 3. DataFrame Construction

In [22]:
iot_records_df = pd.concat([gps_records_df, temp_records_df, rfid_records_df], ignore_index=True)
print(f"Total records retrieved: {len(iot_records_df)}")
print(iot_records_df.dtypes)


Total records retrieved: 866
timestamp            int64
rfid_tag            object
device_id           object
latitude            object
longitude           object
sensor_type         object
temperature_raw    float64
scan_status         object
dtype: object


## 4. Cleaning & Enrichment

In [23]:
iot_cleaned_df = iot_records_df.copy()

# ── Timestamps ───────────────────────────────────────────────────────────────
iot_cleaned_df.rename(columns={'timestamp': 'blockchain_timestamp'}, inplace=True)
iot_cleaned_df['blockchain_timestamp'] = pd.to_datetime(iot_cleaned_df['blockchain_timestamp'], unit='s')

# ── Coordinates ──────────────────────────────────────────────────────────────
iot_cleaned_df['latitude']  = pd.to_numeric(iot_cleaned_df['latitude'],  errors='coerce')
iot_cleaned_df['longitude'] = pd.to_numeric(iot_cleaned_df['longitude'], errors='coerce')

# ── Temperature decode (×10 int → float °C) ─────────────────────────────────
iot_cleaned_df['temperature_c'] = iot_cleaned_df['temperature_raw'] / 10
iot_cleaned_df.drop(columns=['temperature_raw'], inplace=True)

# ── Drop null identity rows ───────────────────────────────────────────────────
iot_cleaned_df.dropna(subset=['rfid_tag', 'sensor_type'], inplace=True)
iot_cleaned_df.reset_index(drop=True, inplace=True)

# ── Sensor timestamp (from iot_data.csv — not stored on chain) ───────────────
iot_source_df = pd.read_csv('../data/5-iot_data_v2.csv')[['rfid_tag', 'device_id', 'timestamp']].rename(
    columns={'timestamp': 'sensor_timestamp'}
)
iot_source_df['sensor_timestamp'] = pd.to_datetime(iot_source_df['sensor_timestamp'])
iot_source_df = iot_source_df.drop_duplicates(subset=['rfid_tag', 'device_id'])

iot_cleaned_df = iot_cleaned_df.merge(iot_source_df, on=['rfid_tag', 'device_id'], how='left')
iot_cleaned_df = iot_cleaned_df[iot_cleaned_df['rfid_tag'] != 'TEST-001']

# ── Reorder: blockchain_timestamp first, then sensor_timestamp ────────────────
cols_front = ['blockchain_timestamp', 'sensor_timestamp']
iot_cleaned_df = iot_cleaned_df[cols_front + [c for c in iot_cleaned_df.columns if c not in cols_front]]

# ── is_flagged boolean ────────────────────────────────────────────────────────
iot_cleaned_df['is_flagged'] = iot_cleaned_df['scan_status'] == 'FLAGGED'

print(f"Cleaned so far: {len(iot_cleaned_df)} rows")


Cleaned so far: 866 rows


In [24]:
# ── goods_category, origin, destination — pull from chain ────────────────────
CATEGORY_INDEX_TO_NAME = {
    0: 'Deep Freeze', 1: 'Frozen', 2: 'Chill/Refrigerated',
    3: 'Pharma', 4: 'Cool-Chain', 5: 'Dry Goods',
    6: 'Electronics', 7: 'Clothing', 8: 'Industrial'
}

TEMP_RANGES = {
    'Deep Freeze':        (-30.0, -28.0),
    'Frozen':             (-20.0, -16.0),
    'Chill/Refrigerated': (2.0,    4.0),
    'Pharma':             (2.0,    8.0),
    'Cool-Chain':         (12.0,  14.0),
}

SAFE_MAX = {k: v[1] for k, v in TEMP_RANGES.items()}

tag_to_category = {}
tag_to_origin   = {}
tag_to_dest     = {}

all_tags = logistics_contract.functions.getAllRFIDTags().call()
for tag in all_tags:
    if tag == 'TEST-001':
        continue
    s = logistics_contract.functions.getShipment(tag).call()
    tag_to_category[tag] = CATEGORY_INDEX_TO_NAME.get(s[1])
    tag_to_origin[tag]   = s[2]
    tag_to_dest[tag]     = s[3]

iot_cleaned_df['goods_category'] = iot_cleaned_df['rfid_tag'].map(tag_to_category)
iot_cleaned_df['origin']         = iot_cleaned_df['rfid_tag'].map(tag_to_origin)
iot_cleaned_df['destination']    = iot_cleaned_df['rfid_tag'].map(tag_to_dest)

print("goods_category sample:", iot_cleaned_df['goods_category'].value_counts().head())


goods_category sample: goods_category
Frozen                167
Pharma                161
Cool-Chain            122
Industrial            100
Chill/Refrigerated     85
Name: count, dtype: int64


In [25]:
# ── temp_breach boolean ───────────────────────────────────────────────────────
def compute_temp_breach(row):
    if row['sensor_type'] != 'Temperature':
        return None
    cat = tag_to_category.get(row['rfid_tag'])
    if cat not in TEMP_RANGES:
        return None
    low, high = TEMP_RANGES[cat]
    return not (low <= row['temperature_c'] <= high)

iot_cleaned_df['temp_breach'] = iot_cleaned_df.apply(compute_temp_breach, axis=1)

# ── breach_delta_c — how far above safe max the temp went ────────────────────
def breach_delta(row):
    if row['sensor_type'] != 'Temperature' or pd.isna(row['temperature_c']):
        return None
    safe_max = SAFE_MAX.get(row['goods_category'])
    if safe_max is None:
        return None
    delta = round(row['temperature_c'] - safe_max, 1)
    return delta if delta > 0 else None

iot_cleaned_df['breach_delta_c'] = iot_cleaned_df.apply(breach_delta, axis=1)

print(f"Temp breaches: {(iot_cleaned_df['temp_breach']==True).sum()}")
print(f"Breach delta non-null: {iot_cleaned_df['breach_delta_c'].notna().sum()}")


Temp breaches: 40
Breach delta non-null: 40


In [26]:
# ── Join shipment_registry.csv for operational + financial columns ────────────
ship_extra = pd.read_csv('../data/1-shipment_registry_v2.csv')[[
    'rfid_tag', 'package_count', 'vehicle_id', 'driver_id',
    'shipment_status', 'scheduled_delivery_dt',
    'actual_delivery_dt', 'delay_reason'
]]
iot_cleaned_df = iot_cleaned_df.merge(ship_extra, on='rfid_tag', how='left')

# ── Computed datetime columns for heatmap & timeline ────────────────────────
iot_cleaned_df['scan_hour'] = pd.to_datetime(iot_cleaned_df['sensor_timestamp']).dt.hour
iot_cleaned_df['scan_day']  = pd.to_datetime(iot_cleaned_df['sensor_timestamp']).dt.date.astype(str)

# ── Delay hours — for CEO on-time KPI ────────────────────────────────────────
iot_cleaned_df['scheduled_delivery_dt'] = pd.to_datetime(iot_cleaned_df['scheduled_delivery_dt'])
iot_cleaned_df['actual_delivery_dt']    = pd.to_datetime(iot_cleaned_df['actual_delivery_dt'])
iot_cleaned_df['delay_hours'] = (
    (iot_cleaned_df['actual_delivery_dt'] - iot_cleaned_df['scheduled_delivery_dt'])
    .dt.total_seconds() / 3600
).round(1)

print(f"Total columns: {len(iot_cleaned_df.columns)}")
print(iot_cleaned_df.columns.tolist())


Total columns: 25
['blockchain_timestamp', 'sensor_timestamp', 'rfid_tag', 'device_id', 'latitude', 'longitude', 'sensor_type', 'scan_status', 'temperature_c', 'is_flagged', 'goods_category', 'origin', 'destination', 'temp_breach', 'breach_delta_c', 'package_count', 'vehicle_id', 'driver_id', 'shipment_status', 'scheduled_delivery_dt', 'actual_delivery_dt', 'delay_reason', 'scan_hour', 'scan_day', 'delay_hours']


In [27]:
# ── Final column order ────────────────────────────────────────────────────────
final_cols = [
    'blockchain_timestamp', 'sensor_timestamp',
    'rfid_tag', 'device_id', 'sensor_type',
    'latitude', 'longitude',
    'scan_status', 'temperature_c',
    'is_flagged', 'temp_breach', 'breach_delta_c',
    'goods_category', 'origin', 'destination',
    'package_count', 'vehicle_id', 'driver_id',
    'shipment_status', 'scheduled_delivery_dt',
    'actual_delivery_dt', 'delay_reason', 'delay_hours',
    'scan_hour', 'scan_day'
]
iot_cleaned_df = iot_cleaned_df[final_cols]

print(f"Final shape: {iot_cleaned_df.shape}")
display(iot_cleaned_df.head())


Final shape: (866, 25)


,blockchain_timestamp,sensor_timestamp,rfid_tag,device_id,sensor_type,latitude,longitude,scan_status,temperature_c,is_flagged,...,package_count,vehicle_id,driver_id,shipment_status,scheduled_delivery_dt,actual_delivery_dt,delay_reason,delay_hours,scan_hour,scan_day
0,2026-06-24 13:19:44,2026-05-03 07:00:00,RFID-020,GPS255,GPS,14.667702,120.978645,NaN,NaN,False,...,49,VH-020,DR-020,Delivered,2026-05-05 14:00:00,2026-05-05 14:00:00,NaN,0.0,7,2026-05-03
1,2026-06-24 13:19:46,2026-05-03 09:00:00,RFID-001,GPS926,GPS,14.644554,121.026468,NaN,NaN,False,...,9,VH-001,DR-001,Delivered,2026-05-05 20:00:00,2026-05-05 20:00:00,NaN,0.0,9,2026-05-03
2,2026-06-24 13:19:47,2026-05-03 09:00:00,RFID-020,GPS411,GPS,14.597148,120.976441,NaN,NaN,False,...,49,VH-020,DR-020,Delivered,2026-05-05 14:00:00,2026-05-05 14:00:00,NaN,0.0,9,2026-05-03
3,2026-06-24 13:19:48,2026-05-03 11:00:00,RFID-020,GPS671,GPS,14.562975,120.955133,NaN,NaN,False,...,49,VH-020,DR-020,Delivered,2026-05-05 14:00:00,2026-05-05 14:00:00,NaN,0.0,11,2026-05-03
4,2026-06-24 13:19:50,2026-05-03 11:00:00,RFID-007,GPS661,GPS,14.671551,120.940095,NaN,NaN,False,...,25,VH-007,DR-007,Delivered,2026-05-05 21:00:00,2026-05-05 21:00:00,NaN,0.0,11,2026-05-03


## 5. RFID Filter

In [29]:
iot_by_rfid = iot_cleaned_df.groupby('rfid_tag')
print(f"Unique shipments (RFID tags): {iot_by_rfid.ngroups}")
print("Groups:", list(iot_by_rfid.groups.keys()))


Unique shipments (RFID tags): 50
Groups: ['RFID-001', 'RFID-002', 'RFID-003', 'RFID-004', 'RFID-005', 'RFID-006', 'RFID-007', 'RFID-008', 'RFID-009', 'RFID-010', 'RFID-011', 'RFID-012', 'RFID-013', 'RFID-014', 'RFID-015', 'RFID-016', 'RFID-017', 'RFID-018', 'RFID-019', 'RFID-020', 'RFID-021', 'RFID-022', 'RFID-023', 'RFID-024', 'RFID-025', 'RFID-026', 'RFID-027', 'RFID-028', 'RFID-029', 'RFID-030', 'RFID-031', 'RFID-032', 'RFID-033', 'RFID-034', 'RFID-035', 'RFID-036', 'RFID-037', 'RFID-038', 'RFID-039', 'RFID-040', 'RFID-041', 'RFID-042', 'RFID-043', 'RFID-044', 'RFID-045', 'RFID-046', 'RFID-047', 'RFID-048', 'RFID-049', 'RFID-050']


## 6. Stats

In [30]:
stats_rows = []
for sensor_type, group in iot_cleaned_df.groupby('sensor_type'):
    if sensor_type == 'Temperature':
        vals = group['temperature_c'].dropna()
        col  = 'temperature_c'
    elif sensor_type == 'GPS':
        vals = group['latitude'].dropna()
        col  = 'latitude'
    else:
        continue
    if not vals.empty:
        stats_rows.append({
            'sensor_type': sensor_type, 'column': col,
            'mean': np.mean(vals), 'min': np.min(vals),
            'max': np.max(vals),   'std': np.std(vals)
        })

sensor_stats_df = pd.DataFrame(stats_rows)
print(sensor_stats_df)


   sensor_type         column       mean       min       max        std
0          GPS       latitude  14.328273   6.86439  18.04722   1.324378
1  Temperature  temperature_c  -3.592917 -29.90000  17.00000  14.992259


## 7. Export

In [31]:
iot_cleaned_df.to_csv('../data/6-iot_cleaned_data_v2.csv', index=False)
sensor_stats_df.to_csv('../data/7-sensor_stats_summary_v2.csv', index=False)

print(f"6-iot_cleaned_data_v2.csv  — {len(iot_cleaned_df)} rows, {len(iot_cleaned_df.columns)} columns")
print(f"7-sensor_stats_summary_v2.csv — {len(sensor_stats_df)} rows")
print("\nColumn list:")
for c in iot_cleaned_df.columns:
    print(f"  {c}")


6-iot_cleaned_data_v2.csv  — 866 rows, 25 columns
7-sensor_stats_summary_v2.csv — 2 rows

Column list:
  blockchain_timestamp
  sensor_timestamp
  rfid_tag
  device_id
  sensor_type
  latitude
  longitude
  scan_status
  temperature_c
  is_flagged
  temp_breach
  breach_delta_c
  goods_category
  origin
  destination
  package_count
  vehicle_id
  driver_id
  shipment_status
  scheduled_delivery_dt
  actual_delivery_dt
  delay_reason
  delay_hours
  scan_hour
  scan_day
